In [2]:
"""
WCGAN (Wasserstein GAN with Gradient Penalty) for IDS Minority Class Augmentation
====================================================================================
Step 2: Now that vanilla GAN baseline confirmed stable training across all 4
classes, this upgrades to WGAN-GP -- expected to give better sample diversity
and stability, especially for U2R (only 198 real samples).

Key differences from vanilla GAN (read this before running):
  1. Discriminator -> "Critic": no longer outputs a 0-1 probability
     (no Sigmoid). Instead outputs a raw real-valued "realness score".
     Higher score = critic thinks sample is more real.
  2. Loss function: Wasserstein loss instead of Binary Cross-Entropy.
     Critic loss = mean(critic(fake)) - mean(critic(real))
     Generator loss = -mean(critic(fake))
     This loss is not bounded 0-1 like BCE, so don't compare its scale
     directly against your vanilla GAN loss curves -- compare the SHAPE
     (smooth vs exploding) instead.
  3. Gradient penalty (GP): enforces a mathematical constraint (1-Lipschitz)
     on the critic by penalizing it if gradients w.r.t. interpolated
     real/fake samples stray from norm 1. This is what makes WGAN-GP
     training much more stable than vanilla GAN.
  4. Critic trains n_critic=5 steps for every 1 generator step (critic
     needs to stay ahead of generator for the Wasserstein distance
     estimate to be meaningful).
  5. No BatchNorm in the critic (breaks the gradient penalty math).
     Generator keeps BatchNorm as before.

Author: Vibha J | GAN Implementation Lead
====================================================================================
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.autograd as autograd
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# ============================================================
# CONFIG -- EDIT THESE PATHS TO MATCH YOUR LOCAL SETUP
# ============================================================
DATA_DIR = "./data"
OUTPUT_DIR = "./wcgan_outputs"
LATENT_DIM = 32
LR = 0.0001                 # WGAN-GP typically wants a lower LR than vanilla GAN
BETAS = (0.5, 0.9)          # standard WGAN-GP Adam betas (not 0.5, 0.999)
N_CRITIC = 5                # critic updates per generator update
LAMBDA_GP = 10               # gradient penalty weight
SEED = 42

# Same per-class epoch/batch config as your vanilla GAN baseline, for
# an apples-to-apples comparison.
CLASS_CONFIG = {
    "U2R":    {"file": "gan_input_U2R.csv",    "batch_size": 16, "epochs": 800},
    "R2L":    {"file": "gan_input_R2L.csv",    "batch_size": 64, "epochs": 500},
    "Probe":  {"file": "gan_input_Probe.csv",  "batch_size": 128, "epochs": 300},
    "DoS":    {"file": "gan_input_DoS.csv",    "batch_size": 256, "epochs": 200},
}

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ============================================================
# MODELS
# ============================================================
class Generator(nn.Module):
    """Same architecture as vanilla GAN baseline -- only the training
    procedure changes for WGAN-GP, not the generator itself."""
    def __init__(self, latent_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(True),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(True),
            nn.Linear(128, output_dim),
            nn.Tanh(),  # matches scaler.pkl's [-1, 1] scaling
        )

    def forward(self, z):
        return self.net(z)


class Critic(nn.Module):
    """Replaces the Discriminator. No Sigmoid, no BatchNorm (gradient
    penalty requires each sample's gradient to be independent of other
    samples in the batch, which BatchNorm violates)."""
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2, True),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2, True),
            nn.Linear(64, 1),
            # No Sigmoid -- raw score output for Wasserstein distance
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# GRADIENT PENALTY
# ============================================================
def compute_gradient_penalty(critic, real_samples, fake_samples, device):
    """Interpolates between real and fake samples, then penalizes the
    critic if the gradient norm at that point strays from 1."""
    bs = real_samples.size(0)
    alpha = torch.rand(bs, 1, device=device).expand_as(real_samples)
    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    critic_interpolates = critic(interpolates)

    grad_outputs = torch.ones(bs, 1, device=device)
    gradients = autograd.grad(
        outputs=critic_interpolates,
        inputs=interpolates,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.view(bs, -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty


# ============================================================
# TRAINING LOOP FOR ONE CLASS
# ============================================================
def train_wcgan_for_class(class_name, filepath, batch_size, epochs):
    print(f"\n{'='*60}\nTraining WCGAN (WGAN-GP) for class: {class_name}\n{'='*60}")

    df = pd.read_csv(filepath)
    data = df.values.astype(np.float32)
    n_samples, n_features = data.shape
    print(f"Loaded {n_samples} samples, {n_features} features")

    if n_samples < batch_size:
        print(f"WARNING: batch_size ({batch_size}) > n_samples ({n_samples}). "
              f"Reducing batch_size to {n_samples}.")
        batch_size = n_samples

    tensor_data = torch.tensor(data)
    dataset = TensorDataset(tensor_data)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    if len(loader) == 0:
        # drop_last=True can empty the loader if n_samples == batch_size exactly
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

    G = Generator(LATENT_DIM, n_features).to(device)
    C = Critic(n_features).to(device)

    opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
    opt_C = torch.optim.Adam(C.parameters(), lr=LR, betas=BETAS)

    g_losses, c_losses = [], []
    step = 0

    for epoch in range(epochs):
        epoch_g_loss, epoch_c_loss = 0.0, 0.0
        n_g_updates, n_c_updates = 0, 0

        for (real_batch,) in loader:
            real_batch = real_batch.to(device)
            bs = real_batch.size(0)

            # ---- Train Critic (n_critic times per generator step) ----
            opt_C.zero_grad()
            z = torch.randn(bs, LATENT_DIM, device=device)
            fake_batch = G(z).detach()

            critic_real = C(real_batch).mean()
            critic_fake = C(fake_batch).mean()
            gp = compute_gradient_penalty(C, real_batch, fake_batch, device)

            loss_c = critic_fake - critic_real + LAMBDA_GP * gp
            loss_c.backward()
            opt_C.step()

            epoch_c_loss += loss_c.item()
            n_c_updates += 1
            step += 1

            # ---- Train Generator every N_CRITIC steps ----
            if step % N_CRITIC == 0:
                opt_G.zero_grad()
                z = torch.randn(bs, LATENT_DIM, device=device)
                fake_batch = G(z)
                loss_g = -C(fake_batch).mean()
                loss_g.backward()
                opt_G.step()

                epoch_g_loss += loss_g.item()
                n_g_updates += 1

        g_losses.append(epoch_g_loss / max(n_g_updates, 1))
        c_losses.append(epoch_c_loss / max(n_c_updates, 1))

        if (epoch + 1) % max(1, epochs // 10) == 0:
            print(f"Epoch [{epoch+1}/{epochs}]  Critic_loss: {c_losses[-1]:.4f}  "
                  f"Gen_loss: {g_losses[-1]:.4f}")

    # ---- Save loss curve ----
    plt.figure(figsize=(8, 4))
    plt.plot(g_losses, label="Generator loss")
    plt.plot(c_losses, label="Critic loss (Wasserstein estimate)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"WCGAN (WGAN-GP) Training Loss - {class_name}")
    plt.legend()
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, f"loss_curve_{class_name}.png")
    plt.savefig(plot_path)
    plt.close()
    print(f"Saved loss curve -> {plot_path}")

    # ---- Generate synthetic samples ----
    G.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, LATENT_DIM, device=device)
        synthetic = G(z).cpu().numpy()

    synthetic_df = pd.DataFrame(synthetic, columns=df.columns)
    synth_path = os.path.join(OUTPUT_DIR, f"synthetic_{class_name}.csv")
    synthetic_df.to_csv(synth_path, index=False)
    print(f"Saved synthetic samples -> {synth_path}")

    # ---- Quick statistical sanity check ----
    real_mean = df.mean()
    synth_mean = synthetic_df.mean()
    mean_diff = (real_mean - synth_mean).abs().mean()
    print(f"Mean absolute difference (real vs synthetic feature means): {mean_diff:.4f}")

    # ---- Save trained generator ----
    model_path = os.path.join(OUTPUT_DIR, f"generator_{class_name}.pt")
    torch.save(G.state_dict(), model_path)
    print(f"Saved generator weights -> {model_path}")

    return {
        "class_name": class_name,
        "n_samples": n_samples,
        "n_features": n_features,
        "final_g_loss": g_losses[-1],
        "final_c_loss": c_losses[-1],
        "mean_abs_diff": mean_diff,
    }


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    results = []
    for class_name, cfg in CLASS_CONFIG.items():
        filepath = os.path.join(DATA_DIR, cfg["file"])
        if not os.path.exists(filepath):
            print(f"SKIPPING {class_name}: file not found at {filepath}")
            continue
        result = train_wcgan_for_class(
            class_name=class_name,
            filepath=filepath,
            batch_size=cfg["batch_size"],
            epochs=cfg["epochs"],
        )
        results.append(result)

    # ---- Summary table ----
    print(f"\n{'='*60}\nSUMMARY -- WCGAN (WGAN-GP) Results\n{'='*60}")
    summary_df = pd.DataFrame(results)
    print(summary_df.to_string(index=False))
    summary_path = os.path.join(OUTPUT_DIR, "wcgan_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\nSaved summary -> {summary_path}")
    print("\nHOW TO READ WCGAN LOSS CURVES (different from vanilla GAN):")
    print("- Critic loss trending toward a stable negative-ish plateau = healthy")
    print("- Watch the SHAPE not the raw numbers -- Wasserstein loss is unbounded,")
    print("  unlike BCE loss which was bounded 0-1 in the vanilla GAN baseline.")
    print("- Smooth curves without wild spikes = stable training")
    print("- Compare mean_abs_diff against baseline_summary.csv from vanilla GAN --")
    print("  lower here means WCGAN produced better quality synthetic samples,")
    print("  especially watch U2R since that's the hardest class (fewest real samples).")

Using device: cpu

Training WCGAN (WGAN-GP) for class: U2R
Loaded 198 samples, 101 features
Epoch [80/800]  Critic_loss: -3.4481  Gen_loss: -6.2611
Epoch [160/800]  Critic_loss: -0.6455  Gen_loss: -7.6938
Epoch [240/800]  Critic_loss: -0.6271  Gen_loss: -6.7628
Epoch [320/800]  Critic_loss: -1.2455  Gen_loss: -1.9821
Epoch [400/800]  Critic_loss: -1.0475  Gen_loss: -2.3776
Epoch [480/800]  Critic_loss: -0.8253  Gen_loss: -2.5044
Epoch [560/800]  Critic_loss: -0.6540  Gen_loss: -2.7052
Epoch [640/800]  Critic_loss: -0.5731  Gen_loss: -3.1091
Epoch [720/800]  Critic_loss: -0.6180  Gen_loss: -2.9933
Epoch [800/800]  Critic_loss: -0.5350  Gen_loss: -2.8884
Saved loss curve -> ./wcgan_outputs\loss_curve_U2R.png
Saved synthetic samples -> ./wcgan_outputs\synthetic_U2R.csv
Mean absolute difference (real vs synthetic feature means): 0.0206
Saved generator weights -> ./wcgan_outputs\generator_U2R.pt

Training WCGAN (WGAN-GP) for class: R2L
Loaded 2985 samples, 101 features
Epoch [50/500]  Criti